# Protocol 0 — HEP Proxy Validation

Simulates predictions Pred 0.A–Pred 0.C from the APGI empirical prerequisite protocol:

* **Pred 0.A** — HEP amplitude correlates with heartbeat discrimination d′ (r > 0.35, replicated r ≥ 0.25)
* **Pred 0.B** — Physostigmine (ACh elevation) increases HEP amplitude ≥ 15 % (Cohen's d ≥ 0.50, BF₁₀ ≥ 100)
* **Pred 0.C** — Anterior insula BOLD tracks HEP amplitude trial-by-trial (r > 0.30 within-participant)

> **Note:** All data are synthetic. This notebook uses the APGI linear proxy
> `HEP₂₅₀₋₄₀₀ = a₀ + a₁·Πⁱ + ε_HEP` (a₁ > 0) from `protocols/protocol_0_hep_proxy_validation.json`.
>
> **EP-0 | Paper 1 | Empirical prerequisite | Depends on: —**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from apgi.core import compute_pi_i_eff, BETA_SM_DEFAULT

## 1 — Protocol 0 parameters

From `apgi_parameters` in `protocols/protocol_0_hep_proxy_validation.json`.

In [ ]:
# Protocol 0 APGI parameters (from protocol_0_hep_proxy_validation.json → apgi_parameters & participants)
A0 = 0.10          # HEP intercept (a.u.)
A1 = 0.80          # HEP slope on Πⁱ (proxy validity requires A1 > 0)
PI_I_RESTING = 1.0 # resting Πⁱ estimate
KAPPA = 100.0
N_PRIMARY   = 50   # n_target per participants block
N_REPLICATE = 30   # replication subsample (n_validation_subsample)
rng = np.random.default_rng(2025)

## 2 — Prediction 0.A: HEP ~ heartbeat discrimination d′

In [ ]:
# Simulate Πⁱ per participant (resting-state HEP session)
M_hat_primary = rng.uniform(0.0, 0.5, N_PRIMARY)
pi_i_primary = np.array([compute_pi_i_eff(PI_I_RESTING, BETA_SM_DEFAULT, float(m)) for m in M_hat_primary])
hep_primary = A0 + A1 * pi_i_primary + rng.normal(0, 0.15, N_PRIMARY)

# Heartbeat discrimination d′ — orthogonal Πⁱ measure
d_prime_primary = 0.60 * pi_i_primary + rng.normal(0, 0.20, N_PRIMARY)

r_primary, p_primary = pearsonr(hep_primary, d_prime_primary)
print(f"Primary sample  N={N_PRIMARY}: r = {r_primary:.3f}, p = {p_primary:.4f}")
print(f"Confirming threshold: r > 0.35  → {'PASS' if r_primary > 0.35 else 'FAIL'}")

# Replication subsample
M_hat_rep = rng.uniform(0.0, 0.5, N_REPLICATE)
pi_i_rep = np.array([compute_pi_i_eff(PI_I_RESTING, BETA_SM_DEFAULT, float(m)) for m in M_hat_rep])
hep_rep = A0 + A1 * pi_i_rep + rng.normal(0, 0.18, N_REPLICATE)
d_prime_rep = 0.60 * pi_i_rep + rng.normal(0, 0.22, N_REPLICATE)
r_rep, _ = pearsonr(hep_rep, d_prime_rep)
print(f"Replication subsample N={N_REPLICATE}: r = {r_rep:.3f}")
print(f"Confirming threshold: r ≥ 0.25  → {'PASS' if r_rep >= 0.25 else 'FAIL'}")

## 3 — Prediction 0.B: Physostigmine elevates HEP amplitude

In [ ]:
N_PHYS = 30  # physostigmine arm participants

# Physostigmine elevates Πⁱ via ACh → higher HEP
pi_i_placebo = np.array([compute_pi_i_eff(PI_I_RESTING, BETA_SM_DEFAULT, float(m))
                          for m in rng.uniform(0.0, 0.3, N_PHYS)])
pi_i_physo   = np.array([compute_pi_i_eff(PI_I_RESTING * 1.25, BETA_SM_DEFAULT, float(m))
                          for m in rng.uniform(0.1, 0.5, N_PHYS)])

hep_placebo = A0 + A1 * pi_i_placebo + rng.normal(0, 0.12, N_PHYS)
hep_physo   = A0 + A1 * pi_i_physo   + rng.normal(0, 0.12, N_PHYS)

pct_increase = (hep_physo.mean() - hep_placebo.mean()) / hep_placebo.mean() * 100
cohens_d = (hep_physo.mean() - hep_placebo.mean()) / np.std(hep_placebo)
print(f"HEP placebo: {hep_placebo.mean():.3f} ± {hep_placebo.std():.3f}")
print(f"HEP physostigmine: {hep_physo.mean():.3f} ± {hep_physo.std():.3f}")
print(f"Increase: {pct_increase:.1f}%   Cohen's d = {cohens_d:.3f}")
print(f"Pred 0.B: ≥15% increase → {'PASS' if pct_increase >= 15 else 'FAIL'}")
print(f"Pred 0.B: d ≥ 0.50      → {'PASS' if cohens_d >= 0.50 else 'FAIL'}")

## 4 — Prediction 0.C: Anterior insula BOLD ~ HEP trial-by-trial

In [ ]:
N_SUBJ = 30
N_TRIALS_FMRI = 60  # resting-state trials per participant

r_within = []
for _ in range(N_SUBJ):
    pi_i_s = max(0.2, rng.normal(PI_I_RESTING, 0.2))
    M_hat_s = rng.uniform(0.0, 0.5, N_TRIALS_FMRI)
    pi_eff_s = np.array([compute_pi_i_eff(pi_i_s, BETA_SM_DEFAULT, float(m)) for m in M_hat_s])
    hep_s = A0 + A1 * pi_eff_s + rng.normal(0, 0.10, N_TRIALS_FMRI)
    # aINS BOLD proportional to HEP after arousal covariate control
    bold_ains = 0.55 * hep_s + rng.normal(0, 0.20, N_TRIALS_FMRI)
    r_s, _ = pearsonr(hep_s, bold_ains)
    r_within.append(r_s)

r_mean = np.mean(r_within)
print(f"Mean within-participant r(HEP, aINS BOLD) = {r_mean:.3f} ± {np.std(r_within):.3f}")
print(f"Confirming threshold: r > 0.30  → {'PASS' if r_mean > 0.30 else 'FAIL'}")

## 5 — Summary visualisation (Pred 0.A–0.C)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Pred 0.A: scatter HEP vs d′
ax = axes[0]
ax.scatter(d_prime_primary, hep_primary, alpha=0.6, color="#2166ac", s=30, label="Primary")
ax.scatter(d_prime_rep, hep_rep, alpha=0.6, color="#9966FF", s=20, marker="^", label="Replication")
m, b = np.polyfit(d_prime_primary, hep_primary, 1)
xs = np.linspace(d_prime_primary.min(), d_prime_primary.max(), 50)
ax.plot(xs, m * xs + b, color="#2166ac", lw=1.5)
ax.set_xlabel("Heartbeat discrimination d′")
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title(f"Pred 0.A  r = {r_primary:.3f} (primary)\nr = {r_rep:.3f} (replication)")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

# Pred 0.B: physostigmine vs placebo
ax = axes[1]
ax.bar(["Placebo", "Physostigmine"],
       [hep_placebo.mean(), hep_physo.mean()],
       yerr=[hep_placebo.std() / np.sqrt(N_PHYS), hep_physo.std() / np.sqrt(N_PHYS)],
       color=["#AAAAAA", "#2166ac"], alpha=0.85, edgecolor="white", width=0.4, capsize=5)
ax.set_ylabel("HEP amplitude (a.u.)")
ax.set_title(f"Pred 0.B  +{pct_increase:.1f}%  d={cohens_d:.2f}")
ax.spines[["top", "right"]].set_visible(False)

# Pred 0.C: within-participant r distribution
ax = axes[2]
ax.hist(r_within, bins=12, color="#00CC99", alpha=0.85, edgecolor="white")
ax.axvline(0.30, ls="--", lw=1.5, color="k", label="threshold r=0.30")
ax.axvline(r_mean, ls="-", lw=2, color="#2166ac", label=f"mean r={r_mean:.3f}")
ax.set_xlabel("Within-participant r(HEP, aINS BOLD)")
ax.set_ylabel("Participants")
ax.set_title("Pred 0.C — aINS BOLD ~ HEP")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 0 — HEP Proxy Validation (Pred 0.A–0.C)", fontsize=12)
plt.tight_layout()
plt.show()